# Analyse des Fichiers des Écritures Comptables (FEC)

## Étude de cas – Numeris

### Objectif

L'objectif de cette étude est de construire une première version d'un pipeline d'analyse des Fichiers des Écritures Comptables (FEC) afin d'identifier automatiquement des opportunités de missions de conseil pour les associés de Numeris.

Le projet suit les étapes suivantes :

1. Ingestion des données
2. Conversion au format Parquet
3. Analyse exploratoire (EDA)
4. Construction des indicateurs métiers
5. Détection des opportunités de missions
6. Restitution sous forme de dashboard

> **Choix techniques :** Python, Polars, Parquet et Plotly.

In [100]:
## 1. Import des bibliothèques
import time
from pathlib import Path

import polars as pl

## 2. Définition des chemins de travail

Afin de rendre le notebook facilement réutilisable, tous les chemins vers les données sont centralisés dans cette section.

In [ ]:
# Définition des chemins de travail

SOURCE_DATA = Path("../Data/source")
PARQUET_DATA = Path("../Data/parquet")

FEC_CSV = SOURCE_DATA / "fec_final_anonyme.csv"
CLIENTS_CSV = SOURCE_DATA / "clients_anonymises_final.csv"

FEC_PARQUET = PARQUET_DATA / "fec_final_anonyme.parquet"

# Création du dossier Parquet
PARQUET_DATA.mkdir(parents=True, exist_ok=True)

## 3. Chargement des fichiers CSV

Les données sont fournies sous la forme de deux fichiers :

- un Fichier des Écritures Comptables (FEC) contenant les écritures comptables ;
- un référentiel des clients.

Dans un premier temps, les fichiers sont chargés au format CSV afin d'être convertis ensuite au format Parquet.

In [102]:
import time

start = time.perf_counter()

# Lecture du FEC
fec = pl.read_csv(
    FEC_CSV,
    infer_schema_length=0
)

# Lecture du référentiel clients
clients = pl.read_csv(
    CLIENTS_CSV,
    infer_schema_length=0
)

csv_loading_time = time.perf_counter() - start

In [103]:
# Vérification du chargement
print(f"Temps de chargement : {csv_loading_time:.2f} secondes")

print(f"\nFEC : {fec.height:,} lignes × {fec.width} colonnes")
print(f"Clients : {clients.height:,} lignes × {clients.width} colonnes")

display(fec.head())

display(clients.head())

Temps de chargement : 0.49 secondes

FEC : 1,308,620 lignes × 19 colonnes
Clients : 3,514 lignes × 3 colonnes


JournalCode,JournalLib,EcritureNum,EcritureDate,CompteNum,CompteLib,CompAuxNum,CompAuxLib,PieceRef,PieceDate,EcritureLib,Debit,Credit,EcritureLet,DateLet,ValidDate,Montantdevise,Idevise,siret_anon
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""BQ1""","""Journal de trésorerie - princi…","""1""","""2024-01-01""","""5121001""","""Qonto - principal""",null,null,"""53KQ9ZEZH4""","""2024-01-01""","""Qonto""","""0""","""2400""",null,null,null,"""-2400""","""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""1""","""2024-01-01""","""6270000009""","""Services bancaires et assimilé…",null,null,"""53KQ9ZEZH4""","""2024-01-01""","""Qonto""","""2000""","""0""",null,null,null,"""2000""","""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""1""","""2024-01-01""","""44566""","""TVA déductible sur autres bien…",null,null,"""53KQ9ZEZH4""","""2024-01-01""","""Qonto""","""400""","""0""",null,null,null,"""400""","""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""2""","""2024-01-01""","""5121001""","""Qonto - principal""",null,null,"""FD0F7BSVG2""","""2024-01-01""","""Qonto""","""0""","""5400""",null,null,null,"""-5400""","""EUR""","""46493912008270"""
"""BQ1""","""Journal de trésorerie - princi…","""2""","""2024-01-01""","""6270000009""","""Services bancaires et assimilé…",null,null,"""FD0F7BSVG2""","""2024-01-01""","""Qonto""","""4500""","""0""",null,null,null,"""4500""","""EUR""","""46493912008270"""


name,forme_juridique,siret_anon
str,str,str
"""Client 1""","""EARL""","""46493912008270"""
"""Client 2""","""SA""","""63745184150281"""
"""Client 3""",null,"""54944667148756"""
"""Client 4""","""COM""","""18634956306766"""
"""Client 5""","""SARL""","""45919695190939"""


Le chargement des données s'est déroulé avec succès.

Le FEC contient **1 308 620 écritures comptables** réparties sur **19 colonnes**, ce qui confirme qu'il s'agit d'un jeu de données volumineux. Cette volumétrie justifie le choix de **Polars** pour le traitement ainsi que la conversion vers le format **Parquet**, plus performant pour les analyses ultérieures.

Le référentiel clients contient **3 514 clients** et sera utilisé par la suite pour enrichir les analyses via une jointure avec les données comptables.

In [104]:
fec.schema

Schema([('JournalCode', String),
        ('JournalLib', String),
        ('EcritureNum', String),
        ('EcritureDate', String),
        ('CompteNum', String),
        ('CompteLib', String),
        ('CompAuxNum', String),
        ('CompAuxLib', String),
        ('PieceRef', String),
        ('PieceDate', String),
        ('EcritureLib', String),
        ('Debit', String),
        ('Credit', String),
        ('EcritureLet', String),
        ('DateLet', String),
        ('ValidDate', String),
        ('Montantdevise', String),
        ('Idevise', String),
        ('siret_anon', String)])

### Analyse du schéma

L'ensemble des colonnes est actuellement chargé avec le type `String`. Ce choix est volontaire et permet de conserver fidèlement les données telles qu'elles sont présentes dans le fichier source, sans dépendre de l'inférence automatique des types.

Cette approche est particulièrement adaptée au FEC, dont certaines colonnes peuvent contenir des valeurs alphanumériques (par exemple des numéros de comptes ou des références de pièces). Elle permet d'éviter des erreurs de lecture et de maîtriser explicitement les conversions de types.

Les colonnes représentant des **dates** (`EcritureDate`, `PieceDate`, `DateLet`, `ValidDate`) ainsi que les **montants** (`Debit`, `Credit`, `Montantdevise`) seront converties dans leurs types respectifs avant l'écriture du fichier au format Parquet.

## 4. Conversion des types de données

Les données étant initialement chargées sous forme de chaînes de caractères (`String`), une conversion explicite est réalisée avant la création du fichier Parquet.

Cette étape permet de disposer de types adaptés aux traitements analytiques :

- les dates sont converties au format `Date` ;
- les montants sont convertis en valeurs numériques (`Float64`) afin de permettre les calculs et les agrégations.

In [105]:
fec = fec.with_columns(
    [
        # Dates
        pl.col("EcritureDate").str.to_date("%Y-%m-%d", strict=False),
        pl.col("PieceDate").str.to_date("%Y-%m-%d", strict=False),
        pl.col("DateLet").str.to_date("%Y-%m-%d", strict=False),
        pl.col("ValidDate").str.to_date("%Y-%m-%d", strict=False),

        # Montants
        pl.col("Debit").cast(pl.Float64, strict=False),
        pl.col("Credit").cast(pl.Float64, strict=False),
        pl.col("Montantdevise").cast(pl.Float64, strict=False),
    ]
)

## 4. Conversion du FEC au format Parquet

Le Fichier des Écritures Comptables (FEC) est initialement fourni au format CSV.

Afin d'améliorer les performances des traitements, il est converti au format **Parquet**, un format de stockage colonnaire particulièrement adapté aux traitements analytiques.

In [ ]:
start = time.perf_counter()

fec.write_parquet(FEC_PARQUET)

parquet_writing_time = time.perf_counter() - start

print(f"Conversion réalisée en {parquet_writing_time:.2f} secondes.")

Conversion réalisée en 0.74 secondes.


Le fichier Parquet a été généré avec succès et servira désormais de source principale pour les traitements suivants.

Cette étape permet d'éviter de relire le fichier CSV à chaque exécution du notebook et améliore significativement les performances des traitements.

## 5. Benchmark : CSV vs Parquet

L'objectif est de comparer les deux formats selon deux critères :

- la taille occupée sur le disque ;
- le temps de lecture.

Cette comparaison permet de justifier le choix du format Parquet comme format de travail pour les étapes suivantes.

In [112]:
# Taille des fichiers
csv_size = FEC_CSV.stat().st_size / (1024 ** 2)
parquet_size = FEC_PARQUET.stat().st_size / (1024 ** 2)


# Temps de lecture du CSV

start = time.perf_counter()

_ = pl.read_csv(
    FEC_CSV,
    infer_schema_length=0
)

csv_read_time = time.perf_counter() - start


# Temps de lecture du Parquet

start = time.perf_counter()

_ = pl.read_parquet(FEC_PARQUET)

parquet_read_time = time.perf_counter() - start


# benchmark
benchmark = pl.DataFrame(
    {
        "Format": ["CSV", "Parquet"],
        "Taille (Mo)": [
            round(csv_size, 2),
            round(parquet_size, 2)
        ],
        "Temps de lecture (s)": [
            round(csv_read_time, 3),
            round(parquet_read_time, 3)
        ],
    }
)

benchmark

Format,Taille (Mo),Temps de lecture (s)
str,f64,f64
"""CSV""",205.86,0.396
"""Parquet""",21.09,0.12


La conversion au format Parquet apporte un gain significatif en termes de stockage et de performances.

Le fichier passe d'environ **206 Mo** à **21 Mo**, soit une réduction de près de **90 %** de sa taille. Le temps de lecture est également amélioré, passant de **0,3 seconde** pour le CSV à **0,1 seconde** pour le Parquet.

Ces résultats confirment l'intérêt d'utiliser le format Parquet comme source principale des traitements analytiques. Plus compact et plus rapide à lire, il est particulièrement adapté aux traitements réalisés avec Polars sur des jeux de données volumineux.

## Conclusion de l'étape 0

Cette première étape a permis de préparer les données pour l'ensemble des traitements à venir.

Les deux fichiers fournis ont été chargés avec succès, les types de données ont été contrôlés et convertis de manière explicite, puis le Fichier des Écritures Comptables (FEC) a été enregistré au format Parquet.

Le benchmark réalisé met en évidence les bénéfices de ce format en termes de stockage et de performances. Le fichier Parquet constituera ainsi la source de référence pour toutes les analyses réalisées dans la suite de cette étude.

Cette étape pose les fondations d'un pipeline de données fiable, reproductible et adapté au traitement d'un volume important d'écritures comptables.